# Diffraction Investigation: Laser Wavelength from Workbook Data

This notebook reconstructs a diffraction-based wavelength measurement from the workbook `lazer quest.xlsx`, using the workbook as the main quantitative source and the lab manual only for brief framing.

The aim is to estimate the wavelength of an unknown fibre-coupled diode laser using diffraction experiments alone, by rebuilding the measured datasets, applying linear fits, and comparing the resulting wavelength estimates across several methods.


## Context and Analysis Strategy

The workbook suggests three main experimental strands:

- single-slit Fraunhofer diffraction,
- multi-slit (grating-like) diffraction maxima and missing-order behaviour,
- a Fresnel analysis using a transformed distance variable.

The common analysis pipeline used throughout the notebook is:

$$
\text{raw positions} \rightarrow \Delta y_n \rightarrow \text{linear fit }(y = mn + c) \rightarrow \lambda(m) \rightarrow \sigma_\lambda.
$$

Because the workbook records screen positions rather than intensities, the main analysis uses **symmetric separations** between corresponding features on opposite sides of the pattern,

$$
\Delta y_n = y_{+n} - y_{-n},
$$

which reduces sensitivity to the exact centre position of the pattern.

Within any one fit, $\sigma_m$ refers to the regression-based uncertainty on the fitted slope. The broader spread between different methods is treated separately as evidence of systematic disagreement, not as the same quantity as $\sigma_m$.

Where workbook formulas are clear, they are reproduced. Where a derivation is not fully documented in the workbook (especially in the Fresnel section), the notebook states this explicitly and avoids overclaiming.


In [ ]:
from pathlib import Path
import math
import re
import zipfile
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook

plt.rcParams.update(
    {
        'figure.dpi': 140,
        'axes.grid': True,
        'grid.alpha': 0.25,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.titlesize': 11,
        'axes.labelsize': 10,
        'legend.fontsize': 8,
        'font.size': 10,
        'lines.linewidth': 1.8,
    }
)

BASE_DIR = Path.cwd()
WORKBOOK_PATH = BASE_DIR / 'lazer quest.xlsx'
MANUAL_PATH = BASE_DIR / 'Part_IB_extended_investigation_manual_2026.pdf'

if not WORKBOOK_PATH.exists():
    fallback = Path('/Users/shanilshah/Desktop/coding/lazer quest/lazer quest.xlsx')
    if fallback.exists():
        WORKBOOK_PATH = fallback
if not MANUAL_PATH.exists():
    fallback = Path('/Users/shanilshah/Desktop/coding/lazer quest/Part_IB_extended_investigation_manual_2026.pdf')
    if fallback.exists():
        MANUAL_PATH = fallback

assert WORKBOOK_PATH.exists(), f'Workbook not found: {WORKBOOK_PATH}'
assert MANUAL_PATH.exists(), f'Manual not found: {MANUAL_PATH}'

print(f'Workbook: {WORKBOOK_PATH}')
print(f'Manual (framing only): {MANUAL_PATH}')
print(f'pandas {pd.__version__} | numpy {np.__version__}')


## Load Workbook and Confirm Sheet Structure

The Excel workbook is treated as a semi-structured lab record, so the code reads specific blocks rather than assuming a single tidy table. The `Fresnel ` sheet name includes a trailing space and must be referenced exactly.


In [ ]:
wb_values = load_workbook(WORKBOOK_PATH, data_only=True)
wb_formulas = load_workbook(WORKBOOK_PATH, data_only=False)

sheet_names = wb_values.sheetnames
print('Sheet names:', sheet_names)
assert sheet_names == ['Single slit', 'Grating', 'Fresnel '], 'Unexpected sheet names or order.'

ws_single_v = wb_values['Single slit']
ws_grating_v = wb_values['Grating']
ws_fresnel_v = wb_values['Fresnel ']

ws_single_f = wb_formulas['Single slit']
ws_grating_f = wb_formulas['Grating']
ws_fresnel_f = wb_formulas['Fresnel ']


## Helper Functions

The workbook mixes raw measurements, cached calculations, and embedded formulas. The helpers below keep extraction, cleaning, pairing, fitting, and plotting logic explicit and reusable.


In [ ]:
def clean_numeric(value):
    """Convert workbook entries to float; known text markers become NaN."""
    if value is None:
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    s = str(value).strip()
    if s == '':
        return np.nan
    if s.lower() in {'missing', 'not measurable'}:
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan


def parse_nominal_uncertainty(text: str):
    """Parse strings like '5.12 +- 0.03 m' or '0.000497 + 0.000002 m'."""
    if text is None:
        raise ValueError('Cannot parse None metadata string')
    s = str(text).strip()
    nums = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', s)
    if not nums:
        raise ValueError(f'No numeric values found in: {s!r}')
    nominal = float(nums[0])
    uncertainty = float(nums[1]) if len(nums) > 1 else None

    unit = None
    unit_match = re.search(r'([A-Za-z][A-Za-z0-9^/ _-]*)\s*$', s)
    if unit_match:
        unit = unit_match.group(1).strip()
    return nominal, uncertainty, unit


def _unit_factor_to_m(unit: str) -> float:
    unit = unit.strip().lower()
    mapping = {'m': 1.0, 'cm': 1e-2, 'mm': 1e-3}
    if unit not in mapping:
        raise ValueError(f'Unsupported unit: {unit}')
    return mapping[unit]


def extract_order_position_block(ws, order_col, pos_col, row_start, row_end, position_unit='cm', dataset=None, err_col=None, err_unit=None):
    records = []
    factor = _unit_factor_to_m(position_unit)
    err_factor = _unit_factor_to_m(err_unit if err_unit else position_unit) if err_col else None
    for r in range(row_start, row_end + 1):
        order_raw = ws[f'{order_col}{r}'].value
        pos_raw = ws[f'{pos_col}{r}'].value
        err_raw = ws[f'{err_col}{r}'].value if err_col else None
        order_num = clean_numeric(order_raw)
        pos_num = clean_numeric(pos_raw)
        err_num = clean_numeric(err_raw) if err_col else np.nan
        pos_m = pos_num * factor if np.isfinite(pos_num) else np.nan
        err_m = err_num * err_factor if err_col and np.isfinite(err_num) else np.nan
        records.append(
            {
                'row': r,
                'order_raw': order_raw,
                'position_raw': pos_raw,
                'position_err_raw': err_raw,
                'order': order_num,
                'position_m': pos_m,
                'position_err_m': err_m,
                'source_sheet': ws.title,
                'dataset': dataset,
            }
        )
    df = pd.DataFrame(records)
    df = df.loc[~(df['order'].isna() & df['position_m'].isna())].reset_index(drop=True)
    return df


def pair_symmetric_orders(df, order_col='order', pos_col='position_m', err_col='position_err_m'):
    """
    Pair +n and -n positions by absolute order.

    Fallback: if no negative orders are present but each positive order appears twice
    (as in the Grating 3 raw block), infer first occurrence = +n and second = -n.
    """
    columns = ['abs_order', 'y_plus_m', 'y_plus_err_m', 'y_minus_m', 'y_minus_err_m', 'delta_y_m', 'delta_y_err_m', 'pairing_mode']
    valid_cols = [order_col, pos_col, 'row']
    if err_col in df.columns:
        valid_cols.append(err_col)
    valid = df[valid_cols].copy()
    if err_col not in valid.columns:
        valid[err_col] = np.nan
    valid = valid[np.isfinite(valid[order_col]) & np.isfinite(valid[pos_col])].copy()
    if valid.empty:
        return pd.DataFrame(columns=columns)

    valid['order_int'] = valid[order_col].round().astype(int)

    pos = valid[valid['order_int'] > 0].copy()
    neg = valid[valid['order_int'] < 0].copy()

    if not neg.empty:
        pos_use = pos[['order_int', pos_col, err_col]].copy().rename(columns={'order_int': 'abs_order', pos_col: 'y_plus_m', err_col: 'y_plus_err_m'})
        neg_use = neg[['order_int', pos_col, err_col]].copy()
        neg_use['abs_order'] = (-neg_use['order_int']).astype(int)
        neg_use = neg_use.rename(columns={pos_col: 'y_minus_m', err_col: 'y_minus_err_m'})[['abs_order', 'y_minus_m', 'y_minus_err_m']]
        paired = pd.merge(pos_use, neg_use, on='abs_order', how='inner')
        paired['pairing_mode'] = 'signed_orders'
    else:
        dup = pos.copy()
        dup['abs_order'] = dup['order_int'].astype(int)
        dup = dup.sort_values(['abs_order', 'row']).copy()
        dup['occurrence'] = dup.groupby('abs_order').cumcount()

        first = dup[dup['occurrence'] == 0][['abs_order', pos_col, err_col]].rename(columns={pos_col: 'y_plus_m', err_col: 'y_plus_err_m'})
        second = dup[dup['occurrence'] == 1][['abs_order', pos_col, err_col]].rename(columns={pos_col: 'y_minus_m', err_col: 'y_minus_err_m'})
        paired = pd.merge(first, second, on='abs_order', how='inner')
        paired['pairing_mode'] = 'unsigned_duplicate_rows'

    paired = paired.sort_values('abs_order').reset_index(drop=True)
    paired['delta_y_m'] = paired['y_plus_m'] - paired['y_minus_m']
    paired['delta_y_err_m'] = np.sqrt(np.square(paired['y_plus_err_m']) + np.square(paired['y_minus_err_m']))
    return paired[columns]


def extract_two_col_numeric_table(ws, x_col, y_col, row_start, row_end, y_unit=None, x_name='x', y_name='y'):
    records = []
    y_factor = _unit_factor_to_m(y_unit) if y_unit else 1.0
    for r in range(row_start, row_end + 1):
        x_raw = ws[f'{x_col}{r}'].value
        y_raw = ws[f'{y_col}{r}'].value
        x_num = clean_numeric(x_raw)
        y_num = clean_numeric(y_raw)
        records.append(
            {
                'row': r,
                f'{x_name}_raw': x_raw,
                f'{y_name}_raw': y_raw,
                x_name: x_num,
                y_name: y_num * y_factor if np.isfinite(y_num) else np.nan,
            }
        )
    return pd.DataFrame(records)


def extract_rowwise_table(ws, columns: Dict[str, str], row_start: int, row_end: int):
    records = []
    for r in range(row_start, row_end + 1):
        rec = {'row': r}
        for name, col in columns.items():
            rec[name] = ws[f'{col}{r}'].value
        records.append(rec)
    return pd.DataFrame(records)


def linear_fit_with_r2(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 2:
        raise ValueError('At least two valid points are required for a linear fit.')

    x_mean = x.mean()
    y_mean = y.mean()
    sxx = np.sum((x - x_mean) ** 2)
    sxy = np.sum((x - x_mean) * (y - y_mean))
    slope = sxy / sxx
    intercept = y_mean - slope * x_mean
    y_hat = slope * x + intercept

    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y_mean) ** 2)
    r2 = np.nan if ss_tot == 0 else 1.0 - ss_res / ss_tot

    if x.size >= 3 and sxx > 0:
        s2 = ss_res / (x.size - 2)
        residual_stderr = math.sqrt(s2)
        slope_stderr = math.sqrt(s2 / sxx)
        intercept_stderr = math.sqrt(s2 * ((1.0 / x.size) + (x_mean ** 2) / sxx))
    else:
        residual_stderr = np.nan
        slope_stderr = np.nan
        intercept_stderr = np.nan

    if x.size > 1 and np.nanmax(x) > np.nanmin(x):
        x_fit = np.linspace(np.nanmin(x), np.nanmax(x), 200)
    else:
        x_fit = x.copy()
    y_fit = slope * x_fit + intercept

    return {
        'x': x,
        'y': y,
        'slope': float(slope),
        'intercept': float(intercept),
        'slope_stderr': float(slope_stderr),
        'intercept_stderr': float(intercept_stderr),
        'residual_stderr': float(residual_stderr),
        'sxx': float(sxx),
        'r2': float(r2),
        'x_fit': x_fit,
        'y_fit': y_fit,
        'n_points': int(x.size),
    }


def format_fit_stats(fit, slope_scale=1.0, intercept_scale=1.0, slope_unit='', intercept_unit='', extra_lines=None):
    lines = [f"slope = {fit['slope'] * slope_scale:.5g}{slope_unit}"]
    slope_stderr = fit.get('slope_stderr', np.nan)
    if np.isfinite(slope_stderr):
        lines.append(f"σ_slope = {slope_stderr * slope_scale:.5g}{slope_unit}")
    lines.extend(
        [
            f"intercept = {fit['intercept'] * intercept_scale:.5g}{intercept_unit}",
            f"R^2 = {fit['r2']:.5f}",
        ]
    )
    if extra_lines:
        lines.extend(extra_lines)
    return '\n'.join(lines)


def plot_linear_fit(ax, x, y, fit, *, title, x_label, y_label, stats_text=None, data_label='Data', fit_label='Linear fit', color=None, marker='o', yerr=None, xerr=None, error_color='0.35', capsize=3):
    if yerr is not None or xerr is not None:
        err_kwargs = {}
        if yerr is not None:
            err_kwargs['yerr'] = yerr
        if xerr is not None:
            err_kwargs['xerr'] = xerr
        ax.errorbar(
            x,
            y,
            fmt=marker,
            label=data_label,
            color=color,
            ecolor=error_color,
            elinewidth=1.0,
            capsize=capsize,
            markersize=4.5,
            linestyle='none',
            alpha=0.9,
            zorder=3,
            **err_kwargs,
        )
    else:
        ax.scatter(x, y, label=data_label, color=color, marker=marker, s=28, zorder=3)
    ax.plot(fit['x_fit'], fit['y_fit'], label=fit_label, color=(color if color else 'C1'), zorder=2)
    ax.set_title(title)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    if stats_text:
        ax.text(
            0.03,
            0.97,
            stats_text,
            transform=ax.transAxes,
            va='top',
            ha='left',
            fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='0.8', alpha=0.9),
        )
    ax.legend(frameon=False)


def combine_quadrature(terms):
    finite_terms = [float(term) for term in terms if np.isfinite(term)]
    if not finite_terms:
        return np.nan
    return float(math.sqrt(sum(term ** 2 for term in finite_terms)))


def compute_lambda_fraunhofer_from_slope(slope_m_per_order, aperture_m, L_m):
    return (aperture_m * slope_m_per_order) / (2 * L_m)


def propagate_lambda_fraunhofer_uncertainty(slope, slope_stderr, aperture_m, L_m, *, aperture_unc_m=np.nan, L_unc_m=np.nan):
    if not np.isfinite(slope_stderr):
        return np.nan
    return combine_quadrature(
        [
            (aperture_m / (2 * L_m)) * slope_stderr if np.isfinite(slope_stderr) else np.nan,
            (slope / (2 * L_m)) * aperture_unc_m if np.isfinite(aperture_unc_m) else np.nan,
            (-(aperture_m * slope) / (2 * (L_m ** 2))) * L_unc_m if np.isfinite(L_unc_m) else np.nan,
        ]
    )


def propagate_lambda_from_slope_over_2L_uncertainty(slope, slope_stderr, L_m, *, L_unc_m=np.nan):
    if not np.isfinite(slope_stderr):
        return np.nan
    return combine_quadrature(
        [
            (1.0 / (2 * L_m)) * slope_stderr if np.isfinite(slope_stderr) else np.nan,
            (-(slope) / (2 * (L_m ** 2))) * L_unc_m if np.isfinite(L_unc_m) else np.nan,
        ]
    )


def propagate_lambda_fresnel_uncertainty(slope_stderr, aperture_m):
    if not np.isfinite(slope_stderr):
        return np.nan
    return float(abs(aperture_m ** 2) * slope_stderr)


def format_lambda_line(lambda_nm, lambda_unc_nm, suffix=''):
    if np.isfinite(lambda_unc_nm):
        base = f"λ = {lambda_nm:.1f} ± {lambda_unc_nm:.1f} nm"
    else:
        base = f"λ = {lambda_nm:.1f} nm"
    return f'{base}{suffix}'


def summarise_fit(method_name, fit, lambda_nm, lambda_unc_nm, notes, fit_equation_native, slope_unit, intercept_unit, uncertainty_model):
    return {
        'Method': method_name,
        'Fit equation (native units)': fit_equation_native,
        'Slope': float(fit['slope']),
        'Slope stderr': float(fit['slope_stderr']),
        'Intercept': float(fit['intercept']),
        'R^2': float(fit['r2']),
        'Wavelength estimate (nm)': float(lambda_nm),
        'Wavelength uncertainty (nm)': float(lambda_unc_nm),
        'Uncertainty model': uncertainty_model,
        'Slope unit': slope_unit,
        'Intercept unit': intercept_unit,
        'n_points': int(fit['n_points']),
        'Notes': notes,
    }


def parse_grating_pitch_mm_from_formula(formula_text):
    s = str(formula_text).replace(' ', '')
    m = re.search(r'\*([0-9]+(?:\.[0-9]+)?)\/\(2\*5\.12\)\*10000', s)
    if not m:
        raise ValueError(f'Could not infer grating pitch from formula: {formula_text}')
    return float(m.group(1))


def parse_fresnel_transform_constant(formula_text):
    s = str(formula_text).replace(' ', '')
    m = re.search(r'1/\(([0-9]+(?:\.[0-9]+)?)-A\d+\)\*1000', s)
    if not m:
        m = re.search(r'\(1/\(([0-9]+(?:\.[0-9]+)?)-A\d+\)\)\*1000', s)
    if not m:
        raise ValueError(f'Could not infer Fresnel transform constant from formula: {formula_text}')
    return float(m.group(1))


def parse_fresnel_aperture_mm_from_formula(formula_text):
    s = str(formula_text).replace(' ', '')
    m = re.search(r'\*\(([0-9]+(?:\.[0-9]+)?)\*10\^\(-3\)\)\^2', s)
    if not m:
        raise ValueError(f'Could not infer Fresnel aperture width from formula: {formula_text}')
    return float(m.group(1))


def find_chart_xml_refs_for_series(workbook_path, x_ref, y_ref):
    hits = []
    with zipfile.ZipFile(workbook_path, 'r') as zf:
        for name in zf.namelist():
            if not (name.startswith('xl/charts/chart') and name.endswith('.xml')):
                continue
            text = zf.read(name).decode('utf-8', errors='ignore')
            refs = re.findall(r'<c:f>(.*?)</c:f>', text)
            if x_ref in refs and y_ref in refs:
                hits.append({'chart_xml': name, 'refs': refs})
    return hits


fit_results_by_method = {}


## Workbook Metadata and Constants

The screen distance and slit widths are parsed from the `Single slit` sheet text entries. Two additional constants are inferred from workbook formulas (grating pitch for the grating-maxima wavelength formula, and the aperture-width scale used in the Fresnel wavelength formula).


In [ ]:
L_text = ws_single_v['D27'].value
thick_text = ws_single_v['D28'].value
medium_text = ws_single_v['D29'].value
thin_text = ws_single_v['D30'].value

L_nominal, L_unc, L_unit = parse_nominal_uncertainty(L_text)
thick_a_m, thick_a_unc, _ = parse_nominal_uncertainty(thick_text)
medium_a_m, medium_a_unc, _ = parse_nominal_uncertainty(medium_text)
thin_a_m, thin_a_unc, _ = parse_nominal_uncertainty(thin_text)

assert L_unit == 'm'
assert np.isclose(L_nominal, 5.12)

single_slit_meta = {
    'thick': {'label': 'Thick slit', 'a_m': thick_a_m, 'a_unc_m': thick_a_unc, 'raw_cols': ('D', 'E'), 'raw_err_col': 'F', 'raw_rows': (2, 24), 'processed_cols': ('D', 'E'), 'processed_rows': (33, 43)},
    'medium': {'label': 'Medium slit', 'a_m': medium_a_m, 'a_unc_m': medium_a_unc, 'raw_cols': ('A', 'B'), 'raw_err_col': 'C', 'raw_rows': (2, 24), 'processed_cols': ('A', 'B'), 'processed_rows': (33, 37)},
    'thin': {'label': 'Thin slit', 'a_m': thin_a_m, 'a_unc_m': thin_a_unc, 'raw_cols': ('G', 'H'), 'raw_err_col': 'I', 'raw_rows': (2, 24), 'processed_cols': ('G', 'H'), 'processed_rows': (33, 34)},
}

grating_formula_U39 = ws_grating_f['U39'].value
grating_pitch_mm_inferred = parse_grating_pitch_mm_from_formula(grating_formula_U39)
grating_pitch_m = grating_pitch_mm_inferred * 1e-3

fresnel_formula_D2 = ws_fresnel_f['D2'].value
fresnel_transform_constant = parse_fresnel_transform_constant(fresnel_formula_D2)

fresnel_formula_H1 = ws_fresnel_f['H1'].value
fresnel_a_mm_inferred = parse_fresnel_aperture_mm_from_formula(fresnel_formula_H1)
fresnel_a_m = fresnel_a_mm_inferred * 1e-3

L_m = L_nominal

metadata_summary = pd.DataFrame(
    [
        {'Quantity': 'Screen distance L', 'Value': L_m, 'Uncertainty': L_unc, 'Unit': 'm', 'Source': 'Single slit!D27'},
        {'Quantity': 'Single slit width (thick)', 'Value': thick_a_m, 'Uncertainty': thick_a_unc, 'Unit': 'm', 'Source': 'Single slit!D28'},
        {'Quantity': 'Single slit width (medium)', 'Value': medium_a_m, 'Uncertainty': medium_a_unc, 'Unit': 'm', 'Source': 'Single slit!D29'},
        {'Quantity': 'Single slit width (thin)', 'Value': thin_a_m, 'Uncertainty': thin_a_unc, 'Unit': 'm', 'Source': 'Single slit!D30'},
        {'Quantity': 'Grating pitch used for maxima method', 'Value': grating_pitch_m, 'Uncertainty': np.nan, 'Unit': 'm', 'Source': 'Inferred from Grating!U39 formula'},
        {'Quantity': 'Fresnel transform constant', 'Value': fresnel_transform_constant, 'Uncertainty': np.nan, 'Unit': '(same units as d)', 'Source': 'Inferred from Fresnel !D2 formula'},
        {'Quantity': 'Fresnel aperture-width scale', 'Value': fresnel_a_m, 'Uncertainty': np.nan, 'Unit': 'm', 'Source': 'Inferred from Fresnel !H1 formula'},
    ]
)

print('Key workbook formulas used for provenance:')
print('  Grating!U39 =', grating_formula_U39)
print('  Fresnel !D2 =', fresnel_formula_D2)
print('  Fresnel !H1 =', fresnel_formula_H1)
metadata_summary


## Basic Cleaning and Parsing Checks

Before extracting the datasets, confirm that text markers and metadata parsing behave as intended.


In [ ]:
assert np.isnan(clean_numeric('missing'))
assert np.isnan(clean_numeric('not measurable '))
assert np.isnan(clean_numeric(''))
assert clean_numeric('14.2') == 14.2

parsed_examples = pd.DataFrame(
    [
        {'raw': L_text, 'parsed': parse_nominal_uncertainty(L_text)},
        {'raw': thick_text, 'parsed': parse_nominal_uncertainty(thick_text)},
        {'raw': medium_text, 'parsed': parse_nominal_uncertainty(medium_text)},
        {'raw': thin_text, 'parsed': parse_nominal_uncertainty(thin_text)},
    ]
)
parsed_examples


## A. Single-Slit Fraunhofer Analysis

For each single slit, the workbook records minima positions on both sides of the central maximum. The notebook reconstructs the symmetric separations directly from the raw position columns.

Using the small-angle Fraunhofer approximation, the working derivation is:

$$
a\sin\theta = n\lambda
$$

$$
y_n \approx L\tan\theta \approx L\sin\theta
$$

$$
y_n \approx \frac{Ln\lambda}{a}
$$

$$
\Delta y_n = y_{+n} - y_{-n} \approx \frac{2Ln\lambda}{a}
$$

$$
\Delta y_n = mn + c,\quad m \approx \frac{2L\lambda}{a}
$$

$$
\lambda = \frac{am}{2L}
$$

The reported wavelength uncertainty follows the same first-order propagation used in the code:

$$
\sigma_\lambda^2 \approx \left(\frac{\partial \lambda}{\partial m}\sigma_m\right)^2 + \left(\frac{\partial \lambda}{\partial a}\sigma_a\right)^2 + \left(\frac{\partial \lambda}{\partial L}\sigma_L\right)^2
$$

with

$$
\frac{\partial \lambda}{\partial m}=\frac{a}{2L},\quad \frac{\partial \lambda}{\partial a}=\frac{m}{2L},\quad \frac{\partial \lambda}{\partial L}=-\frac{am}{2L^2}.
$$

So the quoted uncertainty is the quadrature combination of fit slope uncertainty, slit-width metadata uncertainty, and screen-distance uncertainty.


In [ ]:
single_raw_blocks = {}
single_pairs = {}

for key, meta in single_slit_meta.items():
    order_col, pos_col = meta['raw_cols']
    r0, r1 = meta['raw_rows']
    raw_df = extract_order_position_block(
        ws_single_v,
        order_col=order_col,
        pos_col=pos_col,
        err_col=meta['raw_err_col'],
        row_start=r0,
        row_end=r1,
        position_unit='cm',
        err_unit='cm',
        dataset=meta['label'],
    )
    pair_df = pair_symmetric_orders(raw_df)
    pair_df['dataset'] = meta['label']
    pair_df['a_m'] = meta['a_m']

    single_raw_blocks[key] = raw_df
    single_pairs[key] = pair_df

single_pairs_display = pd.concat(
    [
        df.assign(
            y_plus_cm=df['y_plus_m'] * 100,
            y_minus_cm=df['y_minus_m'] * 100,
            delta_y_cm=df['delta_y_m'] * 100,
        )[['abs_order', 'y_plus_cm', 'y_minus_cm', 'delta_y_cm', 'pairing_mode', 'dataset']]
        for key, df in single_pairs.items()
    ],
    ignore_index=True,
)

single_pairs_display


In [ ]:
# Cross-check reconstructed symmetric separations against workbook cached tables.
for key, meta in single_slit_meta.items():
    px_col, py_col = meta['processed_cols']
    r0, r1 = meta['processed_rows']
    processed = extract_two_col_numeric_table(
        ws_single_v,
        x_col=px_col,
        y_col=py_col,
        row_start=r0,
        row_end=r1,
        y_unit='cm',
        x_name='abs_order',
        y_name='delta_y_m',
    )
    processed = processed[np.isfinite(processed['abs_order']) & np.isfinite(processed['delta_y_m'])].copy()
    processed['abs_order'] = processed['abs_order'].round().astype(int)
    processed = processed.sort_values('abs_order').reset_index(drop=True)

    reconstructed = single_pairs[key][['abs_order', 'delta_y_m']].copy().sort_values('abs_order').reset_index(drop=True)

    assert np.array_equal(reconstructed['abs_order'].to_numpy(), processed['abs_order'].to_numpy())
    assert np.allclose(reconstructed['delta_y_m'].to_numpy(), processed['delta_y_m'].to_numpy(), rtol=0, atol=1e-12)
    print(f"{meta['label']}: reconstructed separations match workbook cached values.")

single_fit_results = {}
for key, pair_df in single_pairs.items():
    meta = single_slit_meta[key]
    fit = linear_fit_with_r2(pair_df['abs_order'], pair_df['delta_y_m'])
    lambda_m = compute_lambda_fraunhofer_from_slope(fit['slope'], meta['a_m'], L_m)
    lambda_nm = lambda_m * 1e9
    lambda_unc_m = propagate_lambda_fraunhofer_uncertainty(
        slope=fit['slope'],
        slope_stderr=fit['slope_stderr'],
        aperture_m=meta['a_m'],
        L_m=L_m,
        aperture_unc_m=meta['a_unc_m'],
        L_unc_m=L_unc,
    )
    lambda_unc_nm = lambda_unc_m * 1e9 if np.isfinite(lambda_unc_m) else np.nan
    single_fit_results[key] = {'fit': fit, 'lambda_nm': lambda_nm, 'lambda_unc_nm': lambda_unc_nm}

    fit_results_by_method[f"Single slit ({key})"] = summarise_fit(
        method_name=f"Single slit ({key})",
        fit=fit,
        lambda_nm=lambda_nm,
        lambda_unc_nm=lambda_unc_nm,
        notes=f"Fraunhofer minima; a = {meta['a_m']*1e3:.3f} mm; uncertainty includes fit slope, slit-width metadata, and L",
        fit_equation_native='Δy (m) = m n + c',
        slope_unit='m/order',
        intercept_unit='m',
        uncertainty_model='Quadrature of fit slope, slit-width metadata, and screen-distance L',
    )

single_fit_summary_df = pd.DataFrame(
    [
        {
            'Slit': single_slit_meta[k]['label'],
            'a (mm)': single_slit_meta[k]['a_m'] * 1e3,
            'Slope (cm/order)': single_fit_results[k]['fit']['slope'] * 100,
            'Slope stderr (cm/order)': single_fit_results[k]['fit']['slope_stderr'] * 100,
            'Intercept (cm)': single_fit_results[k]['fit']['intercept'] * 100,
            'R^2': single_fit_results[k]['fit']['r2'],
            'Lambda (nm)': single_fit_results[k]['lambda_nm'],
            'Lambda uncertainty (nm)': single_fit_results[k]['lambda_unc_nm'],
        }
        for k in ['medium', 'thick', 'thin']
    ]
)

single_fit_summary_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2), constrained_layout=True)
plot_order = ['medium', 'thick', 'thin']
colors = {'medium': 'C0', 'thick': 'C1', 'thin': 'C2'}

for ax, key in zip(axes, plot_order):
    pair_df = single_pairs[key]
    fit = single_fit_results[key]['fit']
    fit_cm = dict(fit)
    fit_cm['y_fit'] = fit['y_fit'] * 100

    stats_text = format_fit_stats(
        fit,
        slope_scale=100,
        intercept_scale=100,
        slope_unit=' cm/order',
        intercept_unit=' cm',
        extra_lines=[format_lambda_line(single_fit_results[key]['lambda_nm'], single_fit_results[key]['lambda_unc_nm'])],
    )

    plot_linear_fit(
        ax,
        x=pair_df['abs_order'],
        y=pair_df['delta_y_m'] * 100,
        yerr=pair_df['delta_y_err_m'] * 100,
        fit=fit_cm,
        title=single_slit_meta[key]['label'],
        x_label='Order n',
        y_label='Symmetric separation Δy (cm)',
        stats_text=stats_text,
        data_label='Reconstructed pairs',
        fit_label='Linear fit',
        color=colors[key],
    )

fig.suptitle('Single-slit Fraunhofer analysis from reconstructed symmetric minima separations', y=1.03)
plt.show()


The medium and thick slit datasets are strongly linear, with very high $R^2$ values and enough points to support a stable fit. The thin slit dataset contains only two paired orders, so its fitted line is exact by construction and should be treated as a weak estimate. With only two points there is no reliable regression-based $\sigma_m$, which is why no propagated wavelength uncertainty is reported for that case.


## B. Combined Scaled Single-Slit Analysis (Preferred Single-Slit Summary)

The workbook also forms a scaled variable proportional to $a\,\Delta y_n$. Reconstructing this directly from the raw paired data gives a single consistency test across all slit widths:

$$
a\,\Delta y_n = 2L\lambda n
$$

$$
a\,\Delta y_n = mn + c,\quad m \approx 2L\lambda
$$

$$
\lambda = \frac{m}{2L}
$$

$$
\sigma_\lambda^2 \approx \left(\frac{1}{2L}\sigma_m\right)^2 + \left(-\frac{m}{2L^2}\sigma_L\right)^2.
$$

If the single-slit datasets are mutually consistent, all points should collapse onto one line. In this pooled form the notebook only propagates the fitted slope uncertainty and the screen-distance uncertainty.


In [ ]:
combined_scaled_records = []
for key in ['thick', 'medium', 'thin']:
    pair_df = single_pairs[key]
    a_m = single_slit_meta[key]['a_m']
    tmp = pair_df.copy()
    tmp['dataset_key'] = key
    tmp['dataset'] = single_slit_meta[key]['label']
    tmp['a_m'] = a_m
    tmp['a_delta_y_m2'] = tmp['a_m'] * tmp['delta_y_m']
    tmp['a_delta_y_err_m2'] = tmp['a_m'] * tmp['delta_y_err_m']
    combined_scaled_records.append(tmp)

combined_scaled_df = pd.concat(combined_scaled_records, ignore_index=True)
combined_scaled_fit = linear_fit_with_r2(combined_scaled_df['abs_order'], combined_scaled_df['a_delta_y_m2'])
combined_scaled_lambda_m = combined_scaled_fit['slope'] / (2 * L_m)
combined_scaled_lambda_nm = combined_scaled_lambda_m * 1e9
combined_scaled_lambda_unc_m = propagate_lambda_from_slope_over_2L_uncertainty(
    slope=combined_scaled_fit['slope'],
    slope_stderr=combined_scaled_fit['slope_stderr'],
    L_m=L_m,
    L_unc_m=L_unc,
)
combined_scaled_lambda_unc_nm = combined_scaled_lambda_unc_m * 1e9 if np.isfinite(combined_scaled_lambda_unc_m) else np.nan

fit_results_by_method['Single slit (combined scaled)'] = summarise_fit(
    method_name='Single slit (combined scaled)',
    fit=combined_scaled_fit,
    lambda_nm=combined_scaled_lambda_nm,
    lambda_unc_nm=combined_scaled_lambda_unc_nm,
    notes='Pooled a·Δy vs n across thick/medium/thin slits; uncertainty includes fit slope and L',
    fit_equation_native='aΔy (m^2) = m n + c',
    slope_unit='m^2/order',
    intercept_unit='m^2',
    uncertainty_model='Quadrature of fit slope and screen-distance L',
)

# Optional cross-check against workbook mixed-unit scaled table (column E is in cm·m).
workbook_scaled = extract_rowwise_table(ws_single_v, {'n_raw': 'D', 'scaled_cm_m_raw': 'E'}, 46, 63)
workbook_scaled['n'] = workbook_scaled['n_raw'].apply(clean_numeric)
workbook_scaled['scaled_cm_m'] = workbook_scaled['scaled_cm_m_raw'].apply(clean_numeric)
workbook_scaled['a_delta_y_m2'] = workbook_scaled['scaled_cm_m'] * 1e-2  # convert cm·m to m^2
workbook_scaled_valid = workbook_scaled[np.isfinite(workbook_scaled['n']) & np.isfinite(workbook_scaled['a_delta_y_m2'])].copy()
workbook_scaled_valid['n'] = workbook_scaled_valid['n'].astype(int)

reconstructed_multiset = sorted(zip(combined_scaled_df['abs_order'].astype(int), np.round(combined_scaled_df['a_delta_y_m2'], 12)))
workbook_multiset = sorted(zip(workbook_scaled_valid['n'].astype(int), np.round(workbook_scaled_valid['a_delta_y_m2'], 12)))
assert reconstructed_multiset == workbook_multiset, 'Combined scaled reconstruction differs from workbook cached table.'

assert 660 <= combined_scaled_lambda_nm <= 680, f'Unexpected combined single-slit wavelength: {combined_scaled_lambda_nm:.2f} nm'

combined_scaled_df[['dataset', 'abs_order', 'a_m', 'delta_y_m', 'a_delta_y_m2']]


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.8), constrained_layout=True)
palette = {'Thick slit': 'C1', 'Medium slit': 'C0', 'Thin slit': 'C2'}

for dataset, sub in combined_scaled_df.groupby('dataset', sort=False):
    ax.errorbar(
        sub['abs_order'],
        sub['a_delta_y_m2'],
        yerr=sub['a_delta_y_err_m2'],
        fmt='o',
        label=dataset,
        color=palette.get(dataset),
        ecolor='0.35',
        elinewidth=1.0,
        capsize=3,
        markersize=4.5,
        alpha=0.9,
        zorder=3,
    )

ax.plot(combined_scaled_fit['x_fit'], combined_scaled_fit['y_fit'], color='black', label='Global linear fit', zorder=2)
ax.set_title('Combined scaled single-slit analysis')
ax.set_xlabel('Order n')
ax.set_ylabel(r'$a\,\Delta y$ (m$^2$)')
ax.legend(frameon=False)
ax.text(
    0.03,
    0.97,
    format_fit_stats(
        combined_scaled_fit,
        slope_scale=1.0,
        intercept_scale=1.0,
        slope_unit=' m$^2$/order',
        intercept_unit=' m$^2$',
        extra_lines=[format_lambda_line(combined_scaled_lambda_nm, combined_scaled_lambda_unc_nm)],
    ),
    transform=ax.transAxes,
    va='top',
    ha='left',
    fontsize=8.5,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='0.8', alpha=0.9),
)
plt.show()


This pooled fit is the strongest single-slit summary in the notebook because it checks both the expected linear dependence on $n$ and the expected inverse scaling with slit width $a$.


## C. Grating Maxima Analysis (workbook-equivalent and Pooled Sensitivity)

The `Grating` sheet contains six raw maxima datasets and a processed block of symmetric separations. The workbook's embedded grating-maxima wavelength formula corresponds to a fit based on **Grating 1** (the workbook chart uses the Grating 1 processed series), so that result is reproduced as the workbook-equivalent estimate.

$$
d_g\sin\theta = n\lambda
$$

$$
\Delta y_n \approx \frac{2Ln\lambda}{d_g}
$$

$$
\Delta y_n = mn + c,\quad \lambda = \frac{d_g m}{2L}.
$$

A second fit pools all valid grating datasets as a sensitivity/consistency check. The notebook's quoted uncertainty follows the same slope-and-$L$ propagation pattern as the combined scaled single-slit fit, but it treats the inferred grating pitch $d_g$ as exact, so it should be read as a lower-bound uncertainty rather than a full calibration uncertainty.


In [ ]:
grating_specs = [
    {'key': 'g1', 'label': 'Grating 1', 'cols': ('A', 'B'), 'rows': (3, 14), 'processed': ('A', 'B', 17, 22), 'processed_err_col': 'C'},
    {'key': 'g2', 'label': 'Grating 2', 'cols': ('E', 'F'), 'rows': (3, 14), 'processed': ('E', 'F', 17, 22), 'processed_err_col': 'G'},
    {'key': 'g3', 'label': 'Grating 3', 'cols': ('H', 'I'), 'rows': (3, 14), 'processed': ('H', 'I', 17, 22), 'processed_err_col': 'J'},
    {'key': 'g4', 'label': 'Grating 4', 'cols': ('K', 'L'), 'rows': (3, 14), 'processed': ('K', 'L', 17, 22), 'processed_err_col': 'M'},
    {'key': 'g5', 'label': 'Grating 5', 'cols': ('N', 'O'), 'rows': (3, 14), 'processed': ('N', 'O', 17, 22), 'processed_err_col': 'P'},
    {'key': 'g6', 'label': 'Grating 6', 'cols': ('Q', 'R'), 'rows': (3, 14), 'processed': ('Q', 'R', 17, 22), 'processed_err_col': 'S'},
]

grating_raw_blocks = {}
grating_pairs = {}
for spec in grating_specs:
    raw_df = extract_order_position_block(
        ws_grating_v,
        order_col=spec['cols'][0],
        pos_col=spec['cols'][1],
        row_start=spec['rows'][0],
        row_end=spec['rows'][1],
        position_unit='cm',
        dataset=spec['label'],
    )
    pair_df = pair_symmetric_orders(raw_df)
    proc = extract_rowwise_table(
        ws_grating_v,
        {'abs_order_raw': spec['processed'][0], 'delta_y_cm_raw': spec['processed'][1], 'delta_y_err_cm_raw': spec['processed_err_col']},
        spec['processed'][2],
        spec['processed'][3],
    )
    proc['abs_order'] = proc['abs_order_raw'].apply(clean_numeric)
    proc['delta_y_m'] = proc['delta_y_cm_raw'].apply(clean_numeric) * 1e-2
    proc['delta_y_err_m'] = proc['delta_y_err_cm_raw'].apply(clean_numeric) * 1e-2
    proc = proc[np.isfinite(proc['abs_order']) & np.isfinite(proc['delta_y_m'])].copy()
    proc['abs_order'] = proc['abs_order'].astype(int)
    proc = proc[['abs_order', 'delta_y_m', 'delta_y_err_m']].sort_values('abs_order').reset_index(drop=True)
    pair_df = pair_df.sort_values('abs_order').reset_index(drop=True)
    assert np.array_equal(proc['abs_order'].to_numpy(), pair_df['abs_order'].to_numpy())
    assert np.allclose(proc['delta_y_m'].to_numpy(), pair_df['delta_y_m'].to_numpy(), rtol=0, atol=1e-12)
    pair_df['delta_y_err_m'] = pair_df['abs_order'].map(proc.set_index('abs_order')['delta_y_err_m'])
    assert np.all(np.isfinite(pair_df['delta_y_err_m']))
    pair_df['dataset'] = spec['label']
    grating_raw_blocks[spec['key']] = raw_df
    grating_pairs[spec['key']] = pair_df

# Validate Grating 1 reconstruction against workbook processed values (required check).
g1_processed = extract_two_col_numeric_table(ws_grating_v, 'A', 'B', 17, 22, y_unit='cm', x_name='abs_order', y_name='delta_y_m')
g1_processed = g1_processed[np.isfinite(g1_processed['abs_order']) & np.isfinite(g1_processed['delta_y_m'])].copy()
g1_processed['abs_order'] = g1_processed['abs_order'].astype(int)
g1_processed = g1_processed.sort_values('abs_order').reset_index(drop=True)

g1_reconstructed = grating_pairs['g1'][['abs_order', 'delta_y_m']].copy().sort_values('abs_order').reset_index(drop=True)
assert np.array_equal(g1_processed['abs_order'].to_numpy(), g1_reconstructed['abs_order'].to_numpy())
assert np.allclose(g1_processed['delta_y_m'].to_numpy(), g1_reconstructed['delta_y_m'].to_numpy(), rtol=0, atol=1e-12)
print('Grating 1: reconstructed separations match workbook processed values.')

# Optional broader check: processed grating blocks vs reconstructed pairs for all six datasets.
for spec in grating_specs:
    px, py, r0, r1 = spec['processed']
    proc = extract_two_col_numeric_table(ws_grating_v, px, py, r0, r1, y_unit='cm', x_name='abs_order', y_name='delta_y_m')
    proc = proc[np.isfinite(proc['abs_order']) & np.isfinite(proc['delta_y_m'])].copy()
    proc['abs_order'] = proc['abs_order'].astype(int)
    proc = proc[['abs_order', 'delta_y_m']].sort_values('abs_order').reset_index(drop=True)

    rec = grating_pairs[spec['key']][['abs_order', 'delta_y_m']].sort_values('abs_order').reset_index(drop=True)
    assert np.array_equal(proc['abs_order'].to_numpy(), rec['abs_order'].to_numpy())
    assert np.allclose(proc['delta_y_m'].to_numpy(), rec['delta_y_m'].to_numpy(), rtol=0, atol=1e-12)

print('All reconstructed grating separations agree with workbook cached processed values.')

# Inspect pairing modes to highlight the Grating 3 unsigned-order issue.
pairing_mode_summary = pd.concat(
    [
        grating_pairs[spec['key']][['abs_order', 'pairing_mode']].assign(dataset=spec['label'])
        for spec in grating_specs
    ],
    ignore_index=True,
)

pairing_mode_summary.groupby(['dataset', 'pairing_mode']).size().rename('n_pairs').reset_index()


In [ ]:
# Workbook chart provenance for the workbook-equivalent grating-maxima fit.
chart_hits = find_chart_xml_refs_for_series(
    WORKBOOK_PATH,
    x_ref='Grating!$A$17:$A$22',
    y_ref='Grating!$B$17:$B$22',
)
assert chart_hits, 'Expected to find a chart using Grating!A17:A22 vs B17:B22.'
print('Chart XML files using Grating 1 processed series:')
for hit in chart_hits:
    print(' ', hit['chart_xml'])

# workbook-equivalent (Grating 1) fit.
g1_df = grating_pairs['g1'].copy()
g1_fit = linear_fit_with_r2(g1_df['abs_order'], g1_df['delta_y_m'])
g1_lambda_m = compute_lambda_fraunhofer_from_slope(g1_fit['slope'], grating_pitch_m, L_m)
g1_lambda_nm = g1_lambda_m * 1e9
g1_lambda_unc_m = propagate_lambda_fraunhofer_uncertainty(
    slope=g1_fit['slope'],
    slope_stderr=g1_fit['slope_stderr'],
    aperture_m=grating_pitch_m,
    L_m=L_m,
    L_unc_m=L_unc,
)
g1_lambda_unc_nm = g1_lambda_unc_m * 1e9 if np.isfinite(g1_lambda_unc_m) else np.nan

fit_results_by_method['Grating maxima (workbook-equivalent, Grating 1)'] = summarise_fit(
    method_name='Grating maxima (workbook-equivalent, Grating 1)',
    fit=g1_fit,
    lambda_nm=g1_lambda_nm,
    lambda_unc_nm=g1_lambda_unc_nm,
    notes=f'Uses Grating 1 processed series; grating pitch inferred as {grating_pitch_m*1e3:.3f} mm from Grating!U39; uncertainty includes fit slope and L only (grating pitch treated as exact)',
    fit_equation_native='Δy (m) = m n + c',
    slope_unit='m/order',
    intercept_unit='m',
    uncertainty_model='Quadrature of fit slope and screen-distance L (grating pitch treated as exact)',
)

# Pooled sensitivity fit across all valid grating datasets.
grating_pooled_df = pd.concat([grating_pairs[s['key']].assign(dataset=s['label']) for s in grating_specs], ignore_index=True)
grating_pooled_fit = linear_fit_with_r2(grating_pooled_df['abs_order'], grating_pooled_df['delta_y_m'])
grating_pooled_lambda_m = compute_lambda_fraunhofer_from_slope(grating_pooled_fit['slope'], grating_pitch_m, L_m)
grating_pooled_lambda_nm = grating_pooled_lambda_m * 1e9
grating_pooled_lambda_unc_m = propagate_lambda_fraunhofer_uncertainty(
    slope=grating_pooled_fit['slope'],
    slope_stderr=grating_pooled_fit['slope_stderr'],
    aperture_m=grating_pitch_m,
    L_m=L_m,
    L_unc_m=L_unc,
)
grating_pooled_lambda_unc_nm = grating_pooled_lambda_unc_m * 1e9 if np.isfinite(grating_pooled_lambda_unc_m) else np.nan

fit_results_by_method['Grating maxima (pooled sensitivity)'] = summarise_fit(
    method_name='Grating maxima (pooled sensitivity)',
    fit=grating_pooled_fit,
    lambda_nm=grating_pooled_lambda_nm,
    lambda_unc_nm=grating_pooled_lambda_unc_nm,
    notes='All valid grating maxima pairs pooled across six gratings (sensitivity/consistency check); uncertainty includes fit slope and L only (grating pitch treated as exact)',
    fit_equation_native='Δy (m) = m n + c',
    slope_unit='m/order',
    intercept_unit='m',
    uncertainty_model='Quadrature of fit slope and screen-distance L (grating pitch treated as exact)',
)

assert 678 <= g1_lambda_nm <= 686, f'Unexpected workbook-equivalent grating wavelength: {g1_lambda_nm:.2f} nm'
assert 668 <= grating_pooled_lambda_nm <= 677, f'Unexpected pooled grating wavelength: {grating_pooled_lambda_nm:.2f} nm'

pd.DataFrame(
    [
        {
            'Result': 'workbook-equivalent (Grating 1)',
            'Slope (cm/order)': g1_fit['slope'] * 100,
            'Slope stderr (cm/order)': g1_fit['slope_stderr'] * 100,
            'Intercept (cm)': g1_fit['intercept'] * 100,
            'R^2': g1_fit['r2'],
            'Lambda (nm)': g1_lambda_nm,
            'Lambda uncertainty (nm)': g1_lambda_unc_nm,
        },
        {
            'Result': 'Pooled sensitivity (all gratings)',
            'Slope (cm/order)': grating_pooled_fit['slope'] * 100,
            'Slope stderr (cm/order)': grating_pooled_fit['slope_stderr'] * 100,
            'Intercept (cm)': grating_pooled_fit['intercept'] * 100,
            'R^2': grating_pooled_fit['r2'],
            'Lambda (nm)': grating_pooled_lambda_nm,
            'Lambda uncertainty (nm)': grating_pooled_lambda_unc_nm,
        },
    ]
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), constrained_layout=True)

# Left panel: workbook-equivalent Grating 1 fit.
g1_fit_cm = dict(g1_fit)
g1_fit_cm['y_fit'] = g1_fit['y_fit'] * 100
plot_linear_fit(
    axes[0],
    x=g1_df['abs_order'],
    y=g1_df['delta_y_m'] * 100,
    yerr=g1_df['delta_y_err_m'] * 100,
    fit=g1_fit_cm,
    title='Grating maxima (workbook-equivalent: Grating 1)',
    x_label='Order n',
    y_label='Symmetric separation Δy (cm)',
    stats_text=format_fit_stats(
        g1_fit,
        slope_scale=100,
        intercept_scale=100,
        slope_unit=' cm/order',
        intercept_unit=' cm',
        extra_lines=[format_lambda_line(g1_lambda_nm, g1_lambda_unc_nm)],
    ),
    data_label='Grating 1 pairs',
    fit_label='Linear fit',
    color='C0',
)

# Right panel: pooled overlay and pooled fit.
palette = plt.cm.tab10(np.linspace(0, 1, len(grating_specs)))
for c, spec in zip(palette, grating_specs):
    sub = grating_pairs[spec['key']]
    axes[1].errorbar(
        sub['abs_order'],
        sub['delta_y_m'] * 100,
        yerr=sub['delta_y_err_m'] * 100,
        fmt='o',
        label=spec['label'],
        color=c,
        ecolor='0.35',
        elinewidth=1.0,
        capsize=3,
        markersize=4.5,
        alpha=0.9,
        zorder=3,
    )
axes[1].plot(grating_pooled_fit['x_fit'], grating_pooled_fit['y_fit'] * 100, color='black', label='Pooled linear fit', zorder=2)
axes[1].set_title('Grating maxima pooled across all valid datasets')
axes[1].set_xlabel('Order n')
axes[1].set_ylabel('Symmetric separation Δy (cm)')
axes[1].legend(frameon=False, ncol=2, fontsize=7.5)
axes[1].text(
    0.03,
    0.97,
    format_fit_stats(
        grating_pooled_fit,
        slope_scale=100,
        intercept_scale=100,
        slope_unit=' cm/order',
        intercept_unit=' cm',
        extra_lines=[format_lambda_line(grating_pooled_lambda_nm, grating_pooled_lambda_unc_nm)],
    ),
    transform=axes[1].transAxes,
    va='top',
    ha='left',
    fontsize=8.3,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='0.8', alpha=0.9),
)

plt.show()


The workbook-equivalent grating result is intentionally kept separate from the pooled fit. The pooled fit is useful as a consistency check, but it is not the same quantity as the workbook's reported grating-maxima wavelength (which is tied to the Grating 1 chart series).


## D. Missing-Order / Envelope Analysis

Rows `25:30` of the `Grating` sheet tabulate slit width, inverse slit width, and a measured missing-order separation. The code uses the following linearisation:

$$
\Delta y_{\text{miss}} \propto \frac{\lambda L}{a}
$$

$$
\Delta y_{\text{miss}} = m\left(\frac{1}{a}\right) + c
$$

$$
m \approx 2L\lambda
$$

$$
\lambda = \frac{m}{2L}
$$

$$
\sigma_\lambda^2 \approx \left(\frac{1}{2L}\sigma_m\right)^2 + \left(-\frac{m}{2L^2}\sigma_L\right)^2.
$$

This gives another wavelength estimate, but note that slit-width uncertainty also affects the horizontal axis through $1/a$. The current reported $\sigma_\lambda$ does not propagate that x-axis calibration term.


In [ ]:
missing_order_raw = extract_rowwise_table(
    ws_grating_v,
    columns={
        'grating_no': 'A',
        'slit_width_mm_raw': 'C',
        'inv_slit_width_cached_raw': 'D',
        'slit_width_err_mm_raw': 'E',
        'missing_sep_m_raw': 'F',
        'missing_sep_err_cm_raw': 'H',
    },
    row_start=25,
    row_end=30,
)

missing_order_df = missing_order_raw.copy()
missing_order_df['grating_no'] = missing_order_df['grating_no'].apply(clean_numeric)
missing_order_df['slit_width_mm'] = missing_order_df['slit_width_mm_raw'].apply(clean_numeric)
missing_order_df['slit_width_m'] = missing_order_df['slit_width_mm'] * 1e-3
missing_order_df['inv_slit_width_cached_m^-1'] = missing_order_df['inv_slit_width_cached_raw'].apply(clean_numeric)
missing_order_df['inv_slit_width_m^-1'] = 1.0 / missing_order_df['slit_width_m']
missing_order_df['missing_sep_m'] = missing_order_df['missing_sep_m_raw'].apply(clean_numeric)
missing_order_df['missing_sep_err_cm'] = missing_order_df['missing_sep_err_cm_raw'].apply(clean_numeric)
missing_order_df['missing_sep_err_m'] = missing_order_df['missing_sep_err_cm'] * 1e-2

# Check consistency of cached and recomputed inverse widths where available.
mask_inv = np.isfinite(missing_order_df['inv_slit_width_cached_m^-1']) & np.isfinite(missing_order_df['inv_slit_width_m^-1'])
assert np.allclose(
    missing_order_df.loc[mask_inv, 'inv_slit_width_cached_m^-1'],
    missing_order_df.loc[mask_inv, 'inv_slit_width_m^-1'],
    rtol=0,
    atol=1e-9,
)

missing_order_fit_df = missing_order_df[np.isfinite(missing_order_df['missing_sep_m'])].copy().reset_index(drop=True)
missing_order_fit = linear_fit_with_r2(missing_order_fit_df['inv_slit_width_m^-1'], missing_order_fit_df['missing_sep_m'])
missing_order_lambda_m = missing_order_fit['slope'] / (2 * L_m)
missing_order_lambda_nm = missing_order_lambda_m * 1e9
missing_order_lambda_unc_m = propagate_lambda_from_slope_over_2L_uncertainty(
    slope=missing_order_fit['slope'],
    slope_stderr=missing_order_fit['slope_stderr'],
    L_m=L_m,
    L_unc_m=L_unc,
)
missing_order_lambda_unc_nm = missing_order_lambda_unc_m * 1e9 if np.isfinite(missing_order_lambda_unc_m) else np.nan

fit_results_by_method['Grating missing-order'] = summarise_fit(
    method_name='Grating missing-order',
    fit=missing_order_fit,
    lambda_nm=missing_order_lambda_nm,
    lambda_unc_nm=missing_order_lambda_unc_nm,
    notes='Missing-order separation vs inverse slit width (row with non-measurable separation excluded); uncertainty includes fit slope and L',
    fit_equation_native='Δy_miss (m) = m (1/a) + c',
    slope_unit='m^2',
    intercept_unit='m',
    uncertainty_model='Quadrature of fit slope and screen-distance L',
)

assert 650 <= missing_order_lambda_nm <= 662, f'Unexpected missing-order wavelength: {missing_order_lambda_nm:.2f} nm'

missing_order_fit_df[
    ['grating_no', 'slit_width_mm', 'inv_slit_width_m^-1', 'missing_sep_m', 'missing_sep_err_m']
]


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
ax.errorbar(
    missing_order_fit_df['inv_slit_width_m^-1'],
    missing_order_fit_df['missing_sep_m'],
    yerr=missing_order_fit_df['missing_sep_err_m'],
    fmt='o',
    capsize=3,
    color='C3',
    ecolor='0.35',
    label='Workbook points (with y-error bars)',
    zorder=3,
)
ax.plot(missing_order_fit['x_fit'], missing_order_fit['y_fit'], color='black', label='Linear fit', zorder=2)
ax.set_title('Missing-order / envelope analysis')
ax.set_xlabel(r'$1/a$ (m$^{-1}$)')
ax.set_ylabel(r'Missing-order separation $\Delta y_{miss}$ (m)')
ax.legend(frameon=False)
ax.text(
    0.03,
    0.97,
    format_fit_stats(
        missing_order_fit,
        slope_scale=1.0,
        intercept_scale=1.0,
        slope_unit=' m$^2$',
        intercept_unit=' m',
        extra_lines=[format_lambda_line(missing_order_lambda_nm, missing_order_lambda_unc_nm)],
    ),
    transform=ax.transAxes,
    va='top',
    ha='left',
    fontsize=8.5,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='0.8', alpha=0.9),
)
plt.show()


This method uses additional physics beyond the ordinary grating-maxima spacing (the single-slit envelope suppressing certain interference maxima), so it acts as a useful independent cross-check rather than just a repetition of the same Fraunhofer fit.


## E. Fresnel Analysis (Workbook Reproduction with Explicit Caveat)

The `Fresnel ` sheet is less self-documenting than the Fraunhofer sections, but it clearly supports a linear fit of a transformed quantity against an integer index $n$.

The notebook therefore reproduces the workbook logic in a workbook-faithful form:

$$
T = \frac{1000}{27.32 - d}
$$

$$
T = mn + c
$$

$$
\lambda = m a^2
$$

$$
\sigma_\lambda \approx a^2 \sigma_m
$$

The pointwise transform uncertainties in the workbook are consistent with:

$$
\sigma_T \approx (\text{fractional error in } d)\,T.
$$

Operationally, the notebook still follows these steps:

- read $d$, `% error`, and $n$,
- reconstruct the transformed variable $1000/(27.32-d)$,
- fit it linearly against $n$,
- reproduce the workbook-style wavelength calculation using the aperture-width scale inferred from `Fresnel !H1`.

This is treated as a **model-sensitive cross-check**. It is a workbook-faithful reconstruction, not a fully recovered first-principles Fresnel derivation.


In [ ]:
fresnel_df = extract_rowwise_table(
    ws_fresnel_v,
    columns={
        'd_raw': 'A',
        'percent_error_raw': 'B',
        'n_raw': 'C',
        'transform_cached_raw': 'D',
        'transform_error_cached_raw': 'E',
    },
    row_start=2,
    row_end=13,
)

fresnel_df['d'] = fresnel_df['d_raw'].apply(clean_numeric)
fresnel_df['percent_error'] = fresnel_df['percent_error_raw'].apply(clean_numeric)
fresnel_df['n'] = fresnel_df['n_raw'].apply(clean_numeric)
fresnel_df['transform_cached'] = fresnel_df['transform_cached_raw'].apply(clean_numeric)
fresnel_df['transform_error_cached'] = fresnel_df['transform_error_cached_raw'].apply(clean_numeric)
fresnel_df['transform_calc'] = 1000.0 / (fresnel_transform_constant - fresnel_df['d'])
fresnel_df['transform_error_calc_from_pct'] = fresnel_df['percent_error'] * fresnel_df['transform_calc']

assert np.allclose(fresnel_df['transform_calc'], fresnel_df['transform_cached'], rtol=0, atol=1e-9)
print('Fresnel transformed variable matches workbook cached values.')

fresnel_fit = linear_fit_with_r2(fresnel_df['n'], fresnel_df['transform_calc'])
fresnel_lambda_m = fresnel_fit['slope'] * (fresnel_a_m ** 2)
fresnel_lambda_nm = fresnel_lambda_m * 1e9
fresnel_lambda_unc_m = propagate_lambda_fresnel_uncertainty(fresnel_fit['slope_stderr'], fresnel_a_m)
fresnel_lambda_unc_nm = fresnel_lambda_unc_m * 1e9 if np.isfinite(fresnel_lambda_unc_m) else np.nan

fit_results_by_method['Fresnel (workbook-style)'] = summarise_fit(
    method_name='Fresnel (workbook-style)',
    fit=fresnel_fit,
    lambda_nm=fresnel_lambda_nm,
    lambda_unc_nm=fresnel_lambda_unc_nm,
    notes='Reproduces workbook transform and wavelength scaling; derivation not fully recoverable from workbook alone; uncertainty includes fit slope only (Fresnel aperture treated as exact)',
    fit_equation_native='1000/(27.32-d) = m n + c',
    slope_unit='(workbook transform units)/order',
    intercept_unit='workbook transform units',
    uncertainty_model='Quadrature of fit slope only (Fresnel aperture treated as exact)',
)

assert 625 <= fresnel_lambda_nm <= 635, f'Unexpected Fresnel wavelength: {fresnel_lambda_nm:.2f} nm'

fresnel_df[['n', 'd', 'percent_error', 'transform_calc', 'transform_error_cached']]


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)
ax.errorbar(
    fresnel_df['n'],
    fresnel_df['transform_calc'],
    yerr=fresnel_df['transform_error_cached'],
    fmt='o',
    capsize=3,
    color='C4',
    ecolor='0.35',
    label='Workbook points (with y-error bars)',
    zorder=3,
)
ax.plot(fresnel_fit['x_fit'], fresnel_fit['y_fit'], color='black', label='Linear fit', zorder=2)
ax.set_title('Fresnel workbook transform vs order')
ax.set_xlabel('Index n')
ax.set_ylabel(r'Workbook transform $1000/(27.32-d)$')
ax.legend(frameon=False)
ax.text(
    0.03,
    0.97,
    format_fit_stats(
        fresnel_fit,
        slope_scale=1.0,
        intercept_scale=1.0,
        slope_unit='',
        intercept_unit='',
        extra_lines=[format_lambda_line(fresnel_lambda_nm, fresnel_lambda_unc_nm, suffix=' (workbook-style)')],
    ),
    transform=ax.transAxes,
    va='top',
    ha='left',
    fontsize=8.5,
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='0.8', alpha=0.9),
)
plt.show()


The Fresnel fit is internally consistent with the workbook, but the geometric meaning of the transformed quantity and the final scaling is not fully explicit from the workbook alone. The result is therefore reported as a cross-check rather than a primary wavelength estimate.


## Comparison Table and Cross-Method Summary

The table below collects the slope, intercept, and $R^2$ for each linear fit, together with the wavelength estimate converted to nm. Grating maxima are reported twice: the workbook-equivalent Grating 1 result and a pooled sensitivity fit.

$$
\text{Slope stderr} = \sigma_m
$$

$$
\text{Wavelength uncertainty} = \sigma_\lambda \text{ from propagated fit/instrument terms}.
$$

That within-method propagated uncertainty should not be confused with the spread between methods, which is a separate cross-method systematic comparison.


In [ ]:
summary_df = pd.DataFrame(list(fit_results_by_method.values()))
method_order = [
    'Single slit (medium)',
    'Single slit (thick)',
    'Single slit (thin)',
    'Single slit (combined scaled)',
    'Grating maxima (workbook-equivalent, Grating 1)',
    'Grating maxima (pooled sensitivity)',
    'Grating missing-order',
    'Fresnel (workbook-style)',
]
summary_df['Method'] = pd.Categorical(summary_df['Method'], categories=method_order, ordered=True)
summary_df = summary_df.sort_values('Method').reset_index(drop=True)

# Preserve a numeric copy for later calculations.
summary_numeric = summary_df.copy()

assert 'Slope stderr' in summary_numeric.columns
assert 'Wavelength uncertainty (nm)' in summary_numeric.columns
assert 'Uncertainty model' in summary_numeric.columns
assert summary_numeric.loc[summary_numeric['n_points'] == 2, 'Slope stderr'].isna().all()
assert summary_numeric.loc[summary_numeric['n_points'] == 2, 'Wavelength uncertainty (nm)'].isna().all()
assert np.all(np.isfinite(summary_numeric.loc[summary_numeric['n_points'] > 2, 'Slope stderr']))
assert np.all(np.isfinite(summary_numeric.loc[summary_numeric['n_points'] > 2, 'Wavelength uncertainty (nm)']))
finite_unc_mask = np.isfinite(summary_numeric['Wavelength uncertainty (nm)'])
assert np.all(summary_numeric.loc[finite_unc_mask, 'Wavelength uncertainty (nm)'] >= 0)
assert np.all(
    summary_numeric.loc[finite_unc_mask, 'Wavelength uncertainty (nm)']
    < summary_numeric.loc[finite_unc_mask, 'Wavelength estimate (nm)']
)

# Format displayed slope/intercept with units embedded in strings.
summary_display = summary_df.copy()
summary_display['Slope'] = summary_display.apply(lambda r: f"{r['Slope']:.6g} {r['Slope unit']}".strip(), axis=1)
summary_display['Slope stderr'] = summary_display.apply(
    lambda r: (f"{r['Slope stderr']:.6g} {r['Slope unit']}".strip() if np.isfinite(r['Slope stderr']) else 'n/a'),
    axis=1,
)
summary_display['Intercept'] = summary_display.apply(lambda r: f"{r['Intercept']:.6g} {r['Intercept unit']}".strip(), axis=1)
summary_display['R^2'] = summary_display['R^2'].map(lambda v: f"{v:.5f}")
summary_display['Wavelength estimate (nm)'] = summary_display['Wavelength estimate (nm)'].map(lambda v: f"{v:.1f}")
summary_display['Wavelength uncertainty (nm)'] = summary_display['Wavelength uncertainty (nm)'].map(
    lambda v: f"{v:.1f}" if np.isfinite(v) else 'n/a'
)
summary_display['n_points'] = summary_display['n_points'].astype(int)

summary_display = summary_display[
    [
        'Method',
        'Fit equation (native units)',
        'Slope',
        'Slope stderr',
        'Intercept',
        'R^2',
        'Wavelength estimate (nm)',
        'Wavelength uncertainty (nm)',
        'n_points',
        'Uncertainty model',
        'Notes',
    ]
]

summary_display


In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8), constrained_layout=True)
plot_df = summary_numeric.copy()
plot_df = plot_df.sort_values('Wavelength estimate (nm)')

y = np.arange(len(plot_df))
finite_unc_mask = np.isfinite(plot_df['Wavelength uncertainty (nm)'])
ax.hlines(y, xmin=plot_df['Wavelength estimate (nm)'].min() - 5, xmax=plot_df['Wavelength estimate (nm)'], color='0.85', linewidth=2)
ax.errorbar(
    plot_df.loc[finite_unc_mask, 'Wavelength estimate (nm)'],
    y[finite_unc_mask],
    xerr=plot_df.loc[finite_unc_mask, 'Wavelength uncertainty (nm)'],
    fmt='o',
    color='C0',
    ecolor='0.35',
    elinewidth=1.0,
    capsize=3,
    markersize=5,
    zorder=3,
)
if (~finite_unc_mask).any():
    ax.scatter(plot_df.loc[~finite_unc_mask, 'Wavelength estimate (nm)'], y[~finite_unc_mask], color='C0', s=45, zorder=3)
ax.set_yticks(y)
ax.set_yticklabels(plot_df['Method'])
ax.set_xlabel('Estimated wavelength (nm)')
ax.set_title('Comparison of wavelength estimates by method')
plt.show()


## Evaluation

It is useful to separate two uncertainty scales in this notebook:

$$
\text{within-method propagated uncertainty}
$$

and

$$
\text{between-method systematic spread}.
$$

The first comes from the regression/instrument terms propagated within a single model. The second comes from the fact that different physically motivated methods do not return exactly the same wavelength.

### Strengths of the analysis

- Symmetric separations $\Delta y_n = y_{+n} - y_{-n}$ reduce sensitivity to the central-position estimate.
- Linear plots make the wavelength extraction transparent and easy to inspect for departures from model assumptions.
- The combined scaled single-slit analysis provides a strong internal consistency check across different slit widths.
- The missing-order analysis uses different diffraction physics from the ordinary maxima-spacing fit, so it acts as a meaningful cross-check.

### Likely systematic errors

- Mixed units in the workbook (m, cm, mm, and transformed Fresnel units) increase the risk of conversion mistakes if handled implicitly.
- Aperture widths and grating dimensions are critical inputs, so any calibration error in these values propagates directly into the wavelength estimate.
- Alignment errors (beam, aperture, screen, and lens alignment) can shift apparent maxima/minima positions.
- Feature locations were likely identified by eye, especially for diffuse minima or weak maxima, which can bias higher-order points.
- For larger angles/orders, the small-angle approximation may begin to matter.

### Which methods are most convincing here

- The **combined scaled single-slit fit** is the strongest single-slit summary because it tests both linearity in order and the expected scaling with slit width.
- The **grating-based Fraunhofer analyses** (maxima spacing and missing-order/envelope analysis) provide a useful pair of cross-checks.
- The **Fresnel analysis** is internally consistent with the workbook but remains less transparent from the workbook alone, so it is treated as a model-sensitive cross-check.

In the end, the between-method systematic spread is what most strongly limits confidence in any single quoted wavelength.


In [ ]:
# Compute a cautious final range from the stronger Fraunhofer/grating methods.
strong_methods = [
    'Single slit (combined scaled)',
    'Grating maxima (workbook-equivalent, Grating 1)',
    'Grating maxima (pooled sensitivity)',
    'Grating missing-order',
]

strong_df = summary_numeric[summary_numeric['Method'].isin(strong_methods)].copy()
strong_vals = strong_df['Wavelength estimate (nm)'].to_numpy(dtype=float)

preferred_low_nm = 5 * math.floor(np.min(strong_vals) / 5)
preferred_high_nm = 5 * math.ceil(np.max(strong_vals) / 5)
combined_central_nm = float(summary_numeric.loc[summary_numeric['Method'] == 'Single slit (combined scaled)', 'Wavelength estimate (nm)'].iloc[0])
fresnel_nm = float(summary_numeric.loc[summary_numeric['Method'] == 'Fresnel (workbook-style)', 'Wavelength estimate (nm)'].iloc[0])

print('Cautious final estimate / range (from stronger Fraunhofer and grating methods):')
print(f'  Preferred range: {preferred_low_nm:.0f} to {preferred_high_nm:.0f} nm')
print(f'  Combined single-slit central estimate: {combined_central_nm:.1f} nm')
print(f'  Fresnel workbook-style cross-check: {fresnel_nm:.1f} nm (reported separately, more model-sensitive)')


## Final Conclusion

The workbook supports several diffraction-based wavelength estimates for the unknown laser, and the stronger Fraunhofer-style analyses place it clearly in the **red** region of the visible spectrum.

A cautious summary is:

- the combined single-slit and grating-based methods cluster in the **mid- to high-600 nm** range,
- the missing-order method gives a compatible but somewhat lower value than the workbook-equivalent Grating 1 maxima fit,
- the Fresnel workbook-style estimate is lower (around the low-630 nm region) and is best treated as a **model-sensitive cross-check** rather than a primary anchor.

Given the spread between methods, the dominant uncertainty is likely systematic (geometry, aperture dimensions, alignment, feature location, and modelling assumptions), so a method-dependent range is more defensible than a single over-precise quoted wavelength.
